In [101]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder
from sklearn.feature_extraction import DictVectorizer

In [74]:
data = pd.read_csv('data/course_lead_Scoring_2026.csv')
data.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


Data preparation

In [75]:
data.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [76]:
data['number_of_courses_viewed'].dtypes

dtype('int64')

In [77]:
cat_cols = list(data.dtypes[data.dtypes == 'string'].index)
data[cat_cols] = data[cat_cols].fillna('NA')

num_cols = list(data.dtypes[data.dtypes != 'string'].index)
data[num_cols] = data[num_cols].fillna(0)

data.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

Question 1
What is the most frequent observation (mode) for the column industry?

- NA
- technology
- healthcare
- retail

In [78]:
print(f" Most frequent observation in the column industry: {data['industry'].mode()[0]}")

 Most frequent observation in the column industry: technology


Question 2
Create the correlation matrix for the numerical features of your dataset. In a correlation matrix, you compute the correlation coefficient between every pair of features.

What are the two features that have the biggest correlation?

- interaction_count and lead_score
- number_of_courses_viewed and lead_score
- number_of_courses_viewed and interaction_count
- annual_income and interaction_count
Only consider the pairs above when answering this question.

In [79]:
corr = data.corr(numeric_only=True)
corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
annual_income,1.000000,0.161300,0.122842,0.229496,0.189652
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204,0.395038
interaction_count,0.122842,0.721609,1.000000,0.915746,0.448624
lead_score,0.229496,0.757204,0.915746,1.000000,0.483246
converted,0.189652,0.395038,0.448624,0.483246,1.000000


In [80]:
corr_pairs = (
    data[num_cols].corr().abs()
    .stack()
    .rename_axis(['feature_1', 'feature_2'])
    .reset_index(name='corr')
)

corr_pairs = corr_pairs[corr_pairs['feature_1'] < corr_pairs['feature_2']].sort_values('corr', ascending=False)

print(f"Highest correlation features are {corr_pairs['feature_1'].iloc[0]} and {corr_pairs['feature_2'].iloc[0]}")

Highest correlation features are interaction_count and lead_score


Split Data

In [81]:
df_full_train, df_test = train_test_split(data, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=42
)

y_train = df_train['converted'].values
y_val = df_val['converted'].values
y_test = df_test['converted'].values

del df_train['converted']
del df_val['converted']
del df_test['converted']

Question 3
Calculate the mutual information score between converted and other categorical variables in the dataset. Use the training set only.
Round the scores to 2 decimals using round(score, 2).
Which of these variables has the biggest mutual information score?

- industry
- location
- lead_source
- employment_status

In [83]:
largestMIscore = 0

for col in cat_cols:
    MI_score = mutual_info_score(y_train, df_train[col])
    print(f"{col} MI score: {round(MI_score, 2)}")
    if MI_score > largestMIscore:
        biggest_feature = col
        largestMIscore = MI_score

print(f"Feautre with biggest MI score is {biggest_feature}")

lead_source MI score: 0.03
industry MI score: 0.0
employment_status MI score: 0.02
location MI score: 0.0
Feautre with biggest MI score is lead_source


Question 4
Now let's train a logistic regression.
Remember that we have several categorical variables in the dataset. Include them using one-hot encoding.
Fit the model on the training dataset.
To make sure the results are reproducible across different versions of Scikit-Learn, fit the model with these parameters:
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000, random_state=42)
Calculate the accuracy on the validation dataset and round it to 2 decimal digits.
What accuracy did you get?

- 0.55
- 0.65
- 0.75
- 0.85

In [127]:
def train_eval(df_train, df_val, y_train, y_val, c=1.0):
	dv = DictVectorizer(sparse=False)
	X_train = dv.fit_transform(df_train.to_dict(orient='records'))
	X_val = dv.transform(df_val.to_dict(orient='records'))
	model = LogisticRegression(solver='liblinear', C=c, max_iter=1000, random_state=42)
	model.fit(X_train, y_train)
	y_pred = model.predict(X_val)
	accuracy = (y_val == (y_pred == 1)).mean()
	return accuracy

baseline_accuracy = train_eval(df_train, df_val, y_train, y_val, c=1.0)
print(f"Accuracy = {round(baseline_accuracy, 2)}")

Accuracy = 0.64


Question 5
Let's find the least useful feature using the feature elimination technique.
Train a model using the same features and parameters as in Q4 (without rounding).
Now exclude each feature from this set and train a model without it. Record the accuracy for each model.
For each feature, calculate the difference between the original accuracy and the accuracy without the feature.
Which of following feature has the smallest difference?

- 'lead_source'
- 'number_of_courses_viewed'
- 'interaction_count'
Note: The difference doesn't have to be positive.

In [131]:
smallest_diff = 1
removed_features = ['lead_source', 'number_of_courses_viewed', 'interaction_count']

for col in removed_features:

	temp_df_train  = df_train.drop(columns=col)
	temp_df_tval   = df_val.drop(columns=col)

	temp_accuracy = train_eval(temp_df_train, temp_df_tval, y_train, y_val, c=1.0)
	diff = abs(baseline_accuracy - temp_accuracy)

	if diff < smallest_diff:
		impactful_feature = col
		smallest_diff = diff

	print(f"Accuracy with {col} removed = {temp_accuracy}, diff = {diff}")

print(f"Feautre with smallest impact on accuracy is {impactful_feature}")

Accuracy with lead_source removed = 0.642, diff = 0.0030000000000000027
Accuracy with number_of_courses_viewed removed = 0.643, diff = 0.0020000000000000018
Accuracy with interaction_count removed = 0.601, diff = 0.04400000000000004
Feautre with smallest impact on accuracy is number_of_courses_viewed


Question 6
Now let's train a regularized logistic regression.
Let's try the following values of the parameter C: [0.000001, 0.00001, 0.0001, 0.001].
Train models using all the features as in Q4.
Calculate the accuracy on the validation dataset and round it to 3 decimal digits.
Which of these C leads to the best accuracy on the validation set?

- 0.000001
- 0.00001
- 0.0001
- 0.001
Note: If there are multiple options, select the smallest C.

In [134]:
bestAcc = 0
C_values= [0.000001, 0.00001, 0.0001, 0.001]

for c in C_values:

	temp_accuracy = train_eval(df_train, df_val, y_train, y_val, c=c)

	if temp_accuracy > bestAcc:
		best_C = c
		bestAcc = temp_accuracy

	print(f"Accuracy with {c} = {temp_accuracy}")

print(f"C with best accuracy is {best_C}")

Accuracy with 1e-06 = 0.598
Accuracy with 1e-05 = 0.598
Accuracy with 0.0001 = 0.613
Accuracy with 0.001 = 0.645
C with best accuracy is 0.001
